# Step 3a — Internet Archive Corpus Building

This notebook downloads OCR text from Internet Archive volumes identified in Steps 2 and 2.5, then searches those texts for mentions of each person in our research corpus.

## Two-tier approach

**Tier 1 — Download:** We download OCR text for volumes in the manifest that either (a) show hits for 2 or more persons in our corpus, or (b) are designated as priority reference works (e.g., biographical dictionaries, railroad directories, contemporary gazetteers). Volumes are sorted by priority tier and persons count to maximise yield per download.

**Tier 2 — Fuzzy name matching:** Because 19th-century OCR is noisy — ligatures misread, 'rn' confused with 'm', '1' with 'l', etc. — simple substring search misses many mentions. We therefore use a two-pass approach:
1. Fast exact substring search (catches clean OCR hits).
2. Sliding-window `partial_ratio` fuzzy match on windows around candidate positions (catches degraded OCR).

For each hit we extract a 500-word context window and store the passage in `texts_downloaded`. All name variants registered for a person (nicknames, maiden names, variant spellings) are searched in addition to the canonical name.

In [2]:
import sys, json, time, re, os
from pathlib import Path
import requests
from tqdm import tqdm
from thefuzz import fuzz

sys.path.insert(0, str(Path('.')))
from db import get_connection, set_progress, pending_persons, log_source, store_text, get_all_persons, name_variants_for

conn = get_connection()
MANIFEST_PATH = Path('ia_corpus_manifest.json')
TEXTS_DIR = Path('ia_texts')  # local cache for downloaded OCR files
TEXTS_DIR.mkdir(exist_ok=True)
STEP = '3a_internet_archive'

## Download helpers

In [3]:
def get_ia_metadata(identifier):
    """Fetch metadata for an IA item. Returns dict with files list."""
    try:
        resp = requests.get(
            f"https://archive.org/metadata/{identifier}",
            headers={"User-Agent": "RailroadTiesPipeline/1.0"},
            timeout=20
        )
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"  Metadata error for {identifier}: {e}")
        return {}


def find_ocr_file(metadata):
    """
    Find the best OCR text file in an IA item's file list.
    Preference order: _djvu.txt > _ocr.txt > .txt
    Returns (filename, download_url) or (None, None).
    """
    identifier = metadata.get("metadata", {}).get("identifier", "")
    files = metadata.get("files", [])

    # Score files by type preference
    candidates = []
    for f in files:
        name = f.get("name", "")
        if name.endswith("_djvu.txt"):
            candidates.append((0, name))
        elif name.endswith("_ocr.txt"):
            candidates.append((1, name))
        elif name.endswith(".txt") and "_" not in name.rsplit(".", 1)[0][-3:]:
            candidates.append((2, name))

    if not candidates:
        return None, None

    best = sorted(candidates)[0][1]
    url = f"https://archive.org/download/{identifier}/{best}"
    return best, url


def clean_ia_identifier(identifier):
    """Strip |hash suffix from IA identifiers (artifact of full-text search API)."""
    return identifier.split("|")[0]


def download_ia_text(identifier, force=False):
    """
    Download OCR text for an IA item. Caches to ia_texts/{identifier}.txt.
    Returns the text content (string) or None on failure.
    """
    identifier = clean_ia_identifier(identifier)
    cache_path = TEXTS_DIR / f"{identifier}.txt"

    if cache_path.exists() and not force:
        return cache_path.read_text(encoding='utf-8', errors='replace')

    metadata = get_ia_metadata(identifier)
    if not metadata:
        return None

    fname, url = find_ocr_file(metadata)
    if not url:
        print(f"  No OCR text found for {identifier}")
        return None

    try:
        # Stream download (OCR files can be 10-50 MB)
        print(f"  Downloading {identifier} ({fname})...")
        resp = requests.get(url, stream=True, timeout=120,
                            headers={"User-Agent": "RailroadTiesPipeline/1.0"})
        resp.raise_for_status()

        content = b""
        for chunk in resp.iter_content(chunk_size=65536):
            content += chunk

        text = content.decode('utf-8', errors='replace')
        cache_path.write_text(text, encoding='utf-8')
        print(f"  Saved {len(text):,} chars to {cache_path.name}")
        return text
    except Exception as e:
        print(f"  Download error for {identifier}: {e}")
        return None

# Batch name matching

Instead of scanning the entire text per-person (O(persons × text_length) with fuzzy sliding windows), we:
1. Build a single compiled regex of all last names + full names (including OCR-error variants)
2. Scan each text **once** with that regex to find all anchor positions
3. At last-name-only anchors, do a small-window fuzzy check for the full name

This is ~100× faster: a 10 MB text that previously took 7+ min per file now takes seconds.

In [4]:
import re
from collections import defaultdict


def _ocr_name_variants(name_lower):
    """
    Generate plausible OCR-degraded variants of a lowered name.
    Applies common 19th-century OCR substitutions one at a time.
    """
    variants = set()
    subs = [
        ('m', 'rn'), ('rn', 'm'),
        ('l', '1'),  ('1', 'l'),
        ('w', 'vv'), ('vv', 'w'),
        ('o', '0'),  ('0', 'o'),
        ('h', 'li'), ('li', 'h'),
    ]
    for old, new in subs:
        if old in name_lower:
            variants.add(name_lower.replace(old, new, 1))
    return variants


def build_person_search_data(persons, conn):
    """
    Pre-compute search data for batch matching.
    Returns (person_entries, lastname_lookup).
    """
    person_entries = []
    lastname_lookup = defaultdict(list)  # lowered_word -> [(rid, entry)]

    for person in persons:
        rid = person["research_id"]
        cname = person["canonical_name"]
        variants = name_variants_for(conn, rid)
        all_names = [cname] + (variants or [])

        full_names = list(set(all_names))
        last_names = set()
        for n in all_names:
            parts = n.split()
            if len(parts) >= 2 and len(parts[-1]) >= 4:
                last_names.add(parts[-1].lower())

        entry = {
            "rid": rid, "canonical": cname,
            "full_names": full_names,
            "full_names_low": [fn.lower() for fn in full_names],
            "last_names": last_names,
        }
        person_entries.append(entry)

        for ln in last_names:
            lastname_lookup[ln].append((rid, entry))
            for v in _ocr_name_variants(ln):
                lastname_lookup[v].append((rid, entry))

    print(f"Search index: {len(person_entries)} persons, "
          f"{len(lastname_lookup)} last-name variants")
    return person_entries, lastname_lookup


_WORD_RE = re.compile(r'[A-Za-z0-9]+')

MAX_MATCHES_PER_PERSON = 50


def search_text_for_all_persons(text, person_entries, lastname_lookup, context_chars=500):
    """
    Search text for all persons at once using word-level last-name anchoring.

    1. Single-pass tokenization → O(1) last-name lookup per word.
    2. At each anchor, check ±200 chars for exact full-name match.
    3. If no exact hit and name ≥ 8 chars, targeted fuzzy check on the small window.
    4. Only store full-name matches (exact or fuzzy); skip lastname-only anchors.
    5. Deduplicate by sorted position (O(n) not O(n²)).
    """
    if not text:
        return []

    text_lower = text.lower()

    # --- Phase 1: tokenize and find last-name anchors ---
    anchors_by_rid = defaultdict(list)
    for m in _WORD_RE.finditer(text_lower):
        word = m.group()
        pos = m.start()
        if word in lastname_lookup:
            for rid, entry in lastname_lookup[word]:
                anchors_by_rid[rid].append((pos, word))

    # --- Phase 2: for each person, check context around anchors ---
    results = []

    for entry in person_entries:
        rid = entry["rid"]
        anchors = anchors_by_rid.get(rid, [])
        if not anchors:
            continue

        # Sort by position for efficient O(n) dedup
        anchors.sort(key=lambda x: x[0])

        matches = []
        last_accepted_pos = -9999

        for pos, anchor_word in anchors:
            if pos - last_accepted_pos < 200:
                continue
            if len(matches) >= MAX_MATCHES_PER_PERSON:
                break

            win_start = max(0, pos - 80)
            win_end = min(len(text_lower), pos + 200)
            window = text_lower[win_start:win_end]

            best = None

            # Exact full-name match in window
            for fn_low, fn_orig in zip(entry["full_names_low"], entry["full_names"]):
                idx = window.find(fn_low)
                if idx >= 0:
                    abs_pos = win_start + idx
                    best = {
                        "position": abs_pos,
                        "match_text": text[abs_pos:abs_pos + len(fn_low)],
                        "score": 100,
                        "method": "exact",
                        "matched_name": fn_orig,
                    }
                    break

            # Fuzzy full-name check (small window only, not entire text)
            if not best:
                for fn_low, fn_orig in zip(entry["full_names_low"], entry["full_names"]):
                    if len(fn_low) >= 8:
                        score = fuzz.partial_ratio(fn_low, window)
                        if score >= 85:
                            best = {
                                "position": pos,
                                "match_text": text[pos:pos + len(anchor_word)],
                                "score": score,
                                "method": "fuzzy",
                                "matched_name": fn_orig,
                            }
                            break

            # Skip lastname-only anchors (too noisy)
            if not best:
                continue

            # Add context
            p = best["position"]
            ml = len(best["match_text"])
            ctx_start = max(0, p - context_chars)
            ctx_end = min(len(text), p + ml + context_chars)
            best["context"] = text[ctx_start:ctx_end]

            matches.append(best)
            last_accepted_pos = pos

        for match in matches:
            results.append((rid, match))

    return results

## Download and index volumes

In [5]:
if not MANIFEST_PATH.exists():
    print(f"ERROR: {MANIFEST_PATH} not found. Run step2_5_corpus_curation.ipynb first.")
else:
    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)

    items = manifest.get("items", [])
    print(f"Manifest loaded: {len(items)} items")

    # Sort by priority then persons_count
    priority_order = {"high": 0, "medium": 1, "low": 2}
    items_sorted = sorted(items, key=lambda x: (priority_order.get(x.get("priority", "medium"), 1),
                                                  -x.get("persons_count", 0)))

    print(f"\nTop 20 items to download:")
    for item in items_sorted[:20]:
        cached = (TEXTS_DIR / f"{item['identifier']}.txt").exists()
        status = "CACHED" if cached else "pending"
        print(f"  [{status:<7}] [{item.get('priority','?'):<6}] p={item.get('persons_count',0):3d}  "
              f"{item['identifier']:<40} {item.get('title','')[:40]}")

Manifest loaded: 211 items

Top 20 items to download:
  [pending] [high  ] p=  5  biennialreportka00kans|4c3b8b98fc821419ffed85df4dd367620d600e29 
  [CACHED ] [high  ] p=  0  americanbio05wilsuoft                    Appletons' cyclopædia of American biogra
  [CACHED ] [high  ] p=  0  bib_fict_1073745_6                       Appletons’ cyclopædia of American biogra
  [CACHED ] [high  ] p=  0  appletonscyclop01wils                    Appletons' cyclopædia of American biogra
  [CACHED ] [high  ] p=  0  cu31924020334813                         The National cyclopædia of American biog
  [CACHED ] [high  ] p=  0  nationalcyclopae08newy                   The National cyclopaedia of American bio
  [CACHED ] [high  ] p=  0  nationalcyclopae00newy                   The National cyclopaedia of American bio
  [CACHED ] [high  ] p=  0  in.ernet.dli.2015.174536                 Dictionary Of American Biography (supple
  [CACHED ] [high  ] p=  0  in.ernet.dli.2015.163981                 Dictionary Of 

In [6]:
DOWNLOAD_DELAY = 3.0  # seconds between downloads (be polite to archive.org)
MAX_DOWNLOADS = 150    # cap per run — increase after verifying things work

downloaded = 0
skipped = 0
failed = 0

for item in tqdm(items_sorted[:MAX_DOWNLOADS]):
    identifier = item.get("identifier")
    if not identifier:
        continue

    identifier = clean_ia_identifier(identifier)
    cache_path = TEXTS_DIR / f"{identifier}.txt"
    if cache_path.exists():
        skipped += 1
        continue

    text = download_ia_text(identifier)
    if text:
        downloaded += 1
    else:
        failed += 1

    time.sleep(DOWNLOAD_DELAY)

print(f"\nDownload complete: {downloaded} new, {skipped} cached, {failed} failed")

  0%|          | 0/150 [00:00<?, ?it/s]

  No OCR text found for pub_editor-publisher


 11%|█▏        | 17/150 [00:04<00:34,  3.85it/s]

  Download error for guidetowisconsin00oehl: 403 Client Error: Forbidden for url: https://ia600409.us.archive.org/2/items/guidetowisconsin00oehl/guidetowisconsin00oehl_djvu.txt


 19%|█▊        | 28/150 [00:11<00:54,  2.23it/s]

...

  Download error for isbn_9781150707780: 403 Client Error: Forbidden for url: https://ia801503.us.archive.org/9/items/isbn_9781150707780/isbn_9781150707780_djvu.txt


 37%|███▋      | 56/150 [01:17<02:50,  1.81s/it]

  Download error for ans-bulletin_1968-04_10: 401 Client Error: Unauthorized for url: https://dn720805.ca.archive.org/0/items/ans-bulletin_1968-04_10/ans-bulletin_1968-04_10_djvu.txt


100%|██████████| 150/150 [01:29<00:00,  1.68it/s]


Download complete: 0 new, 140 cached, 10 failed


## Search downloaded texts for person mentions

In [7]:
# Build search index once
persons = get_all_persons(conn)
person_entries, lastname_lookup = build_person_search_data(persons, conn)

txt_files = list(TEXTS_DIR.glob("*.txt"))
print(f"Searching across {len(txt_files)} cached texts\n")

# IA item → source_id mapping
ia_source_ids = {
    r["item_id"]: r["id"]
    for r in conn.execute(
        "SELECT id, item_id FROM sources_discovered WHERE source_type='internet_archive'"
    ).fetchall()
}

total_passages = 0

for txt_file in tqdm(txt_files):
    identifier = txt_file.stem

    try:
        text = txt_file.read_text(encoding='utf-8', errors='replace')
    except Exception as e:
        print(f"  Read error {identifier}: {e}")
        continue

    source_id = ia_source_ids.get(identifier)
    source_title = ""
    if source_id:
        row = conn.execute("SELECT title FROM sources_discovered WHERE id=?", (source_id,)).fetchone()
        source_title = row["title"] if row else ""

    # Batch search: all persons in a single pass
    matches = search_text_for_all_persons(text, person_entries, lastname_lookup, context_chars=500)

    for rid, p in matches:
        store_text(
            conn, rid,
            passage_text=p["match_text"],
            context_text=p["context"],
            source_id=source_id,
            source_type='internet_archive',
            source_url=f"https://archive.org/details/{identifier}",
            source_title=source_title,
            is_keyword_filtered=0,
            keyword_match=f"name_match:{p['matched_name']}; method:{p['method']}; score:{p['score']}",
            page_num=f"pos:{p['position']}",
        )
        total_passages += 1

print(f"\nTotal passages stored: {total_passages}")

Search index: 570 persons, 986 last-name variants
Searching across 146 cached texts



100%|██████████| 146/146 [01:51<00:00,  1.32it/s]


Total passages stored: 20557


In [8]:
# Mark all persons as done for this step
for person in persons:
    rid = person["research_id"]
    count = conn.execute(
        "SELECT COUNT(*) as n FROM texts_downloaded WHERE research_id=? AND source_type='internet_archive'",
        (rid,)
    ).fetchone()["n"]
    set_progress(conn, rid, STEP, 'done', result_count=count)

print("=== Step 3a Summary ===")
print(f"IA texts cached: {len(list(TEXTS_DIR.glob('*.txt')))}")
print(f"Total passages in DB: {conn.execute('SELECT COUNT(*) as n FROM texts_downloaded WHERE source_type=\"internet_archive\"').fetchone()['n']}")

top_persons = conn.execute("""
    SELECT p.canonical_name, COUNT(*) as passages
    FROM texts_downloaded t JOIN persons p ON t.research_id=p.research_id
    WHERE t.source_type='internet_archive'
    GROUP BY t.research_id ORDER BY passages DESC LIMIT 20
""").fetchall()
print(f"\nTop persons by IA passages:")
for r in top_persons:
    print(f"  {r['canonical_name']:<35} {r['passages']:>4} passages")

=== Step 3a Summary ===
IA texts cached: 146
Total passages in DB: 20557

Top persons by IA passages:
  H. Smith                            4550 passages
  W. Thompson                         1859 passages
  W. Scott                            1670 passages
  O. Palmer                           1066 passages
  E. Wells                            1010 passages
  John T. Smith                        489 passages
  L. Knapp                             440 passages
  Horace Greeley                       427 passages
  W. T. Thompson                       318 passages
  A. C. Phillips                       262 passages
  C. Wilkinson                         249 passages
  J. W. Jackson                        240 passages
  W. H. Roberts                        221 passages
  A. H. Hamilton                       215 passages
  John Sherman                         191 passages
  James Gordon Bennett                 177 passages
  D. R. Anthony                        175 passages
  William A. C